## 1. Install Dependencies

# 🔍 AI Code Reviewer using Microsoft Agent Framework (MAF)

**This notebook demonstrates:**
- Build agents using Microsoft Agent Framework (MAF), AutoGen’s successor  
- Define reusable tools using the `@tool` decorator  
- Compose multiple agents into a workflow graph with conditional routing  
- Understand when workflows outperform a single-agent design  

**Architecture:**

```
          ┌──────────────┐
          │  User Input  │
          └──────┬───────┘
                 │
                 ▼
        ┌──────────────────┐
        │  Router Agent    │
        │ (decides route)  │
        └──────┬───────────┘
               │
     ┌─────────┴─────────┐
     │                   │
     ▼                   ▼
┌──────────────┐   ┌──────────────┐
│ Style Agent  │   │ Security     │
│              │   │ Agent        │
└──────────────┘   └──────────────┘
```

In [63]:
%pip install agent-framework --pre
%pip install python-dotenv flake8 bandit openai

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [61]:
import os
from typing import Annotated
from dotenv import load_dotenv

load_dotenv()

print("✅ API key loaded:", "OPENAI_API_KEY" in os.environ)

✅ API key loaded: True


## 3. Create the MAF Client

MAF supports Azure OpenAI, OpenAI, Anthropic, Ollama, and more.  
We're using plain OpenAI here — no Azure account needed.

In [60]:
from agent_framework.openai import OpenAIChatClient

# Shared client — all agents use this
client = OpenAIChatClient(
    model_id="gpt-4o-mini",
    api_key=os.environ["OPENAI_API_KEY"],
)

print("✅ MAF client ready")

✅ MAF client ready


## 4. Define Tools

Tools are functions decorated with `@tool`. The docstring becomes the tool description the LLM sees.  
MAF uses `approval_mode` to control whether the agent needs human confirmation before calling a tool.

Here we define two analysis tools — one for style, one for security.

In [11]:
from agent_framework import tool
from pydantic import Field


@tool
def analyze_code_style(code: str) -> str:
    import subprocess, tempfile
    
    with tempfile.NamedTemporaryFile(delete=False, suffix=".py") as f:
        f.write(code.encode())
        path = f.name
    
    result = subprocess.run(["flake8", path], capture_output=True, text=True)
    return result.stdout

@tool(approval_mode="never_require")
def analyze_code_security(code: str) -> str:
    import subprocess, tempfile, os

    with tempfile.NamedTemporaryFile(delete=False, suffix=".py") as f:
        f.write(code.encode())
        path = f.name

    result = subprocess.run(
        ["bandit", "-q", "-r", path],
        capture_output=True,
        text=True
    )

    os.remove(path)
    return result.stdout

print("✅ Tools defined")

✅ Tools defined


## 5. Build the Agents

Three agents, each with a focused role:
- **RouterAgent** — reads the user's request, decides which reviewer to call
- **StyleReviewerAgent** — specialist in Python style and code quality
- **SecurityReviewerAgent** — specialist in vulnerabilities and security

Each agent only gets the tools it needs. This is the key MAF pattern.

In [49]:
# Router: decides the review type based on user intent
router_agent = client.as_agent(
    name="RouterAgent",
    instructions="""
You are an intelligent routing agent for a code review system.

Analyze the user's request and determine the PRIMARY intent:

- STYLE → formatting, readability, naming, structure, PEP8, clean code
- SECURITY → vulnerabilities, exploits, injection, secrets, unsafe operations

Rules:
- If the request explicitly mentions security → ROUTE:SECURITY
- If it explicitly mentions style/cleanliness → ROUTE:STYLE
- If the request is general like "review this code" → ROUTE:STYLE
- If BOTH style and security are implied → ROUTE:SECURITY (prefer safety)

Output STRICTLY one of:
ROUTE:STYLE
ROUTE:SECURITY

Do NOT explain. Do NOT add anything else.
"""
)

# Style Reviewer: focused on code quality
style_agent = client.as_agent(
    name="StyleReviewerAgent",
    instructions="""
You are a senior Python code reviewer focused on code quality and maintainability.

Process:
1. Use the analyze_code_style tool to get concrete linting issues.
2. ALSO independently review the code for:
   - readability
   - naming clarity
   - function design
   - edge cases the tool may miss

Then produce a clean, NON-REDUNDANT report.

Output format:

### 1. Issues Found
- Combine tool findings + your own insights
- Avoid repeating the same issue twice
- Mention line numbers if available

### 2. Severity
- Assign: Low / Medium / High
- Be realistic (don’t mark everything High)

### 3. Suggested Fix
- Provide improved code snippets
- Keep them clean and idiomatic
- Focus on best practices

Style guidelines:
- Be concise but insightful
- Avoid generic advice
- Do NOT just restate tool output — interpret it
- Prioritize clarity over verbosity
""",
    tools=[analyze_code_style],
)

# Security Reviewer: focused on vulnerabilities
security_agent = client.as_agent(
    name="SecurityReviewerAgent",
    instructions="""
You are a senior security engineer performing a Python code audit.

Process:
1. Use the analyze_code_security tool to detect known vulnerabilities.
2. ALSO independently analyze the code for:
   - hardcoded secrets
   - unsafe patterns (eval, exec, os.system, subprocess misuse)
   - input validation issues
   - authentication/authorization flaws
   - resource handling issues

IMPORTANT:
- Do NOT rely only on the tool
- Tools miss issues — you must think beyond them

Output format:

### 1. Vulnerabilities Found
For each issue:
- Type (e.g., SQL Injection, Command Injection)
- Description (what is wrong + how it can be exploited)
- Location (line/function if possible)
- Risk Level (Critical / High / Medium / Low)

### 2. Overall Risk
- Summarize overall system risk level

### 3. Remediation
- Provide FIXED code snippets (not just advice)
- Use secure best practices:
  - parameterized queries
  - safe subprocess usage
  - environment variables
  - proper resource handling

Guidelines:
- Be precise, not alarmist
- Avoid duplicate issues
- Prioritize real-world exploitability
- Missing a vulnerability is worse than a false positive
""",
    tools=[analyze_code_security],
)

print("✅ All 3 agents created")

✅ All 3 agents created


## 6. Wire the Workflow Graph

This is where MAF shines. We define **executors** (the agents) and **edges** (the connections between them).  
The router's output determines which branch executes — this is MAF's conditional routing.

```
START → RouterAgent → [ROUTE:STYLE] → StyleReviewerAgent → END
                    → [ROUTE:SECURITY] → SecurityReviewerAgent → END
```

In [54]:
from agent_framework import WorkflowBuilder, AgentExecutor

router_exec = AgentExecutor(router_agent)
style_exec = AgentExecutor(style_agent)
security_exec = AgentExecutor(security_agent)

workflow = (
    WorkflowBuilder(start_executor=router_exec)
    .add_edge(
        router_exec,
        style_exec,
        condition=lambda msg: "ROUTE:STYLE" in msg.agent_response.text
    )
    .add_edge(
        router_exec,
        security_exec,
        condition=lambda msg: "ROUTE:SECURITY" in msg.agent_response.text
    )
    .build()
)

print("✅ Workflow built")

✅ Workflow built


## 7. Run the Code Reviewer

### Test Case 1: Style Review

A messy function with bad naming and missing type hints.

In [56]:
messy_code = '''
def calc(x,y,op):
    if op=="add":
        return x+y
    elif op=="sub":
        return x-y
    elif op=="mul":
        return x*y
    elif op=="div":
        if y==0:
            return None
        return x/y
    else:
        return None
'''

user_request = f"Review my code for style issues:\n{messy_code}"

print("🔄 Running Style Review...\n")
event = await workflow.run(user_request)
outputs = event.get_outputs()

for i, res in enumerate(outputs):
        print(f"\n--- Agent {i+1} Output ---")
        print(res.text)

🔄 Running Style Review...


--- Agent 1 Output ---
ROUTE:STYLE

--- Agent 2 Output ---
### 1. Issues Found
- **Whitespace Issues:** There are several missing whitespaces:
  - After commas in the function definition (line 1).
  - Around operators (lines 2, 4, 6, 8, 9).
- **Function Naming:** The function name `calc` is generic; a more descriptive name like `calculate` would improve clarity.
- **Operation Handling:** The use of multiple `if-elif` statements may lead to less maintenance and readability compared to using a dictionary.
- **Division by Zero Handling:** Returning `None` for division by zero is not informative. Raising an exception would provide clearer feedback.
- **Control Flow Clarity:** The broad fallback in the final `else` returning `None` does not convey information about what went wrong.

### 2. Severity
- **High:** The whitespace issues significantly affect readability and adherence to PEP 8.
- **Medium:** Function naming can lead to misunderstandings about what opera

### Test Case 2: Security Audit

Code with a hardcoded password and SQL injection vulnerability.

In [57]:
vulnerable_code = '''
import sqlite3

DB_PASSWORD = "admin123"
SECRET_KEY = "my-super-secret-key-12345"

def get_user(username):
    conn = sqlite3.connect("users.db")
    cursor = conn.cursor()
    query = "SELECT * FROM users WHERE username = '" + username + "'"
    cursor.execute(query)
    return cursor.fetchone()

def run_command(cmd):
    import os
    os.system(cmd)
'''

user_request = f"Check this code for security vulnerabilities:\n{vulnerable_code}"

print("🔄 Running Security Audit...\n")
event = await workflow.run(user_request)
outputs = event.get_outputs()

for i, res in enumerate(outputs):
        print(f"\n--- Agent {i+1} Output ---")
        print(res.text)

🔄 Running Security Audit...


--- Agent 1 Output ---
ROUTE:SECURITY

--- Agent 2 Output ---
### 1. Vulnerabilities Found

#### Issue 1:
- **Type**: Hardcoded Secrets
- **Description**: Hardcoded password "admin123" could lead to unauthorized access if the source code is exposed. Attackers can read the code and gain access to the database.
- **Location**: Line 3 (`DB_PASSWORD`)
- **Risk Level**: Low

#### Issue 2:
- **Type**: Hardcoded Secrets
- **Description**: Hardcoded secret key "my-super-secret-key-12345" can be exploited if an attacker gains access to the source code, potentially leading to compromised security features that depend on this key.
- **Location**: Line 4 (`SECRET_KEY`)
- **Risk Level**: Low

#### Issue 3:
- **Type**: SQL Injection
- **Description**: The SQL query in `get_user` is constructed using string concatenation, allowing an attacker to inject malicious SQL through the `username` input. This could lead to unauthorized data access or manipulation.
- **Location**:

### Test Case 3: Routing is smart — try your own code

In [58]:
your_code = '''
# Paste your own code here!
def my_function():
    pass
'''

# Try changing the request type:
# "check for security issues" → triggers SecurityReviewerAgent
# "review my code style" → triggers StyleReviewerAgent
user_request = f"Review my code style:\n{your_code}"

event = await workflow.run(user_request)
outputs = event.get_outputs()

for i, res in enumerate(outputs):
        print(f"\n--- Agent {i+1} Output ---")
        print(res.text)


--- Agent 1 Output ---
ROUTE:STYLE

--- Agent 2 Output ---
### 1. Issues Found
- **End of File:** There is a missing newline at the end of the file (line 3). This can lead to inconsistencies and is typically considered poor practice in Python files.
- **Comment Clarity:** The comment `# Paste your own code here!` is not informative as a code comment. It does not describe what the function does, which diminishes the clarity and purpose of the function.

### 2. Severity
- **Low:** The missing newline is a minor issue, though it does affect adherence to best practices.
- **Medium:** The uninformative comment reduces the readability and maintainability of the code.

### 3. Suggested Fix
Here’s a revised version of the code:

```python
def my_function():
    """Placeholder function for future implementation."""
    pass
```

#### Improvements made:
- Added a docstring that succinctly describes the purpose of the function, making it clear that this function is intended for future use.
- Ens

## 8. Why MAF Workflows > a Single Agent

| | Single Agent | MAF Workflow |
|---|---|---|
| **Routing** | Implicit, hidden inside prompt | Explicit graph edges (deterministic) |
| **Specialization** | One agent handles everything | Each agent has a focused role |
| **Observability** | Hard to trace decisions | Step-by-step execution visibility |
| **Scalability** | Prompt grows → messy | Add new nodes cleanly |
| **Control** | Limited control over flow | Full control over execution |
| **Human-in-the-loop** | Hard to inject | Built-in via `approval_mode` |


## 9. When NOT to use MAF

- Simple chatbot with no routing → overkill, use a single agent  
- You already have a working LangGraph setup → no need to switch  
- You need a TypeScript SDK → not supported yet  
- Production-critical system → MAF is still in public preview (APIs may change)
